# 01 - Smart meter producer


In [ ]:
import hashlib, json, math, random, uuid
from base64 import b64encode
from datetime import datetime, timedelta, timezone
from pathlib import Path
import oci

CONFIG_PATH = Path('/Workspace/ORACLE_AIDP_STREAMING/config/runtime.yaml')
config = {}
for raw in CONFIG_PATH.read_text(encoding='utf-8').splitlines():
    line = raw.strip()
    if line and not line.startswith('#') and ':' in line:
        key, value = line.split(':', 1)
        config[key.strip()] = value.strip().strip(chr(34)).strip(chr(39))
required = ('OCI_STREAM_ENDPOINT','OCI_STREAM_OCID','OCI_CONFIG_FILE','OCI_PROFILE','METER_COUNT')
missing = [key for key in required if not config.get(key)]
if missing:
    raise ValueError('runtime.yaml is missing: ' + ', '.join(missing))

def build_event(number, instant):
    start = instant.astimezone(timezone.utc).replace(second=0,microsecond=0) - timedelta(minutes=instant.minute % 15)
    meter_id = f'MTR-{number:06d}'
    rng = random.Random(int(hashlib.sha256(f'42|{meter_id}|{start.isoformat()}'.encode()).hexdigest()[:16],16))
    temperature = 28 + 7 * math.sin((start.hour-13)*math.pi/12) + rng.uniform(-1.5,1.5)
    load = .20 + .55 * max(0,math.sin((start.hour-10)*math.pi/12))
    kwh = round(max(.03,(load+.018*max(0,temperature-30))*(.83 if start.weekday()>=5 else 1)+rng.gauss(0,.045)),4)
    tariff = 'PEAK' if 17 <= start.hour < 22 else 'SHOULDER' if 7 <= start.hour < 17 else 'OFF_PEAK'
    return {'schema_version':'1.0','event_id':str(uuid.uuid5(uuid.UUID('34d45e32-9867-4fcf-b3eb-c5b7fe8b9c28'),f'{meter_id}|{start.isoformat()}')),'event_type':'INTERVAL_READING','event_ts_utc':(start+timedelta(minutes=15,seconds=rng.randint(1,120))).isoformat(),'meter_id':meter_id,'device_id':f'DEV-{number:06d}','service_point_id':f'SP-{number:06d}','service_point_type':'ELECTRIC','interval_start_utc':start.isoformat(),'interval_end_utc':(start+timedelta(minutes=15)).isoformat(),'interval_minutes':15,'consumption_kwh':kwh,'voltage_v':round(230+rng.gauss(0,3),2),'current_a':round(max(0,kwh*4000/230+rng.gauss(0,.2)),3),'power_factor':round(min(1,max(.75,.96+rng.gauss(0,.015))),3),'temperature_c':round(temperature,2),'quality_code':'ACTUAL','tariff_code':tariff,'measurement_events':[],'device_events':[]}

value = config['INTERVAL_START_UTC']
instant = datetime.fromisoformat(value) if value else datetime.now(timezone.utc)
events = [build_event(number,instant) for number in range(1,int(config['METER_COUNT'])+1)]
client = oci.streaming.StreamClient(oci.config.from_file(config['OCI_CONFIG_FILE'],config['OCI_PROFILE']),service_endpoint=config['OCI_STREAM_ENDPOINT'])
entries = [oci.streaming.models.PutMessagesDetailsEntry(key=b64encode(row['meter_id'].encode()).decode(),value=b64encode(json.dumps(row,separators=(',',':')).encode()).decode()) for row in events]
result = client.put_messages(config['OCI_STREAM_OCID'],oci.streaming.models.PutMessagesDetails(messages=entries))
failures = [entry.error_message for entry in result.data.entries if entry.error]
if failures: raise RuntimeError(f'OCI rejected {len(failures)} events: {failures[:3]}')
print(f'Published {len(events)} readings for {instant.isoformat()}')